In [1]:
from methyldl.modelling.dnabert2 import EpigenDnabert2, TrainingArguments
from methyldl.data.dataset import SupervisedDataset
import pandas as pd
import numpy as np
data_path = '/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejectionDMRsStratified/'
from torch import nn
import os

/home/luna.kuleuven.be/u0169940/.cache/pypoetry/virtualenvs/methyldl-GStZGe-R-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dmrs = pd.DataFrame(pd.concat([pd.read_parquet(data_path+f"/{split}.parquet") for split in ["train", "valid", "test"]])["dmr_label"].unique())
dmrs.columns  = ["dmr_label"]

model_instance = EpigenDnabert2(use_cpg_methylation=True, max_sequence_length=150, 
                                trust_remote_code=True, 
                                foundation_model_huggingface="../foundationalModels/DNABERT-2-117M",
                                num_labels=40,
                                num_dmr_labels=max(dmrs["dmr_label"])+1)

/home/luna.kuleuven.be/u0169940/.cache/huggingface/modules/transformers_modules/DNABERT-2-117M/bert_layers.py:135: UserWarning: Using triton implementation of flash attention 2
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../foundationalModels/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ../foundationalModels/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/luna.kuleuven.be/u0169940/Repos/methyldl/methyldl/modelling/dnabert2

In [ ]:
train_dataset = SupervisedDataset(
    tokenizer=model_instance.tokenizer,
    data_path_or_list=os.path.join(data_path, "train"),
    kmer=-1,
    data_interface="pandas",
    lazy_tokenization=False,
    include_dmr_ids=True
)

train_data = pd.read_parquet(data_path + f"/train.parquet")
tokenized_data = model_instance.tokenizer(
    train_data["input_ids"].to_list(),
    return_tensors="pt",
    padding="longest",
    max_length=150,
    truncation=True,
)
tokenized_sequences = tokenized_data["input_ids"]
inversed_vocab = {y: x for (x, y) in model_instance.tokenizer.vocab.items()}

from tqdm import tqdm

inversed_sequnces = list()
for i in tqdm(range(0, len(tokenized_sequences)), desc="Inverting tokenization"):
    inversed_sequnces.append([inversed_vocab[int(x)] for x in tokenized_sequences[i]])

one_cgs_tokens = list()
two_cgs_tokens = list()
three_cgs_tokens = list()

for i in tqdm(range(0, len(tokenized_sequences)), desc="Counting CG patterns"):
    sub = inversed_sequnces[i]
    one_cgs_tokens.append(len([x for x in sub if "CG" in x]))
    two_cgs_tokens.append(len([x for x in sub if "CGCG" in x]))
    three_cgs_tokens.append(len([x for x in sub if "CGCGCG" in x]))

# Analysis of CG preservation in BPE tokens
print("\n" + "=" * 60)
print("DNABERT2 BPE Tokenization Analysis: CG Site Preservation")
print("=" * 60)

pct_with_cg = np.mean(np.array(one_cgs_tokens) > 0) * 100
pct_cg_split = (1 - np.mean(np.array(one_cgs_tokens) > 0)) * 100

print(f"\n[1] CG Dinucleotide Tokenization:")
print(f"    • Reads with ≥1 token containing 'CG': {pct_with_cg:.2f}%")
print(f"    • Reads where CG is split across tokens: {pct_cg_split:.2f}%")

pct_two_cgs = np.mean(np.array(two_cgs_tokens) > 0) * 100
pct_two_cgs_multiple = np.mean(np.array(two_cgs_tokens) > 1) * 100
pct_three_cgs = np.mean(np.array(three_cgs_tokens) > 0) * 100

print(f"\n[2] Multi-CG Token Frequency:")
print(f"    • Reads with ≥1 token containing 'CGCG' (2 consecutive CGs): {pct_two_cgs:.4f}%")
print(f"    • Reads with ≥2 tokens containing 'CGCG': {pct_two_cgs_multiple:.4f}%")
print(f"    • Reads with ≥1 token containing 'CGCGCG' (3 consecutive CGs): {pct_three_cgs:.4f}%")

# Methylation status analysis for CGCG tokens
sub = train_data.loc[np.where(np.array(two_cgs_tokens) > 0)]
identical_count = 0
diff_count = 0

for i in range(0, len(sub)):
    sub_sub = sub.iloc[i]
    pos = sub_sub["input_ids"].find("CGCG")
    methyl_statuses = (sub_sub["methylation_ids"][pos], sub_sub["methylation_ids"][pos + 2])
    if methyl_statuses[0] == methyl_statuses[1]:
        identical_count += 1
    else:
        diff_count += 1

total_cgcg = identical_count + diff_count
pct_identical = (identical_count / total_cgcg) * 100
pct_diff = (diff_count / total_cgcg) * 100

print(f"\n[3] Methylation Status Concordance in CGCG Tokens (n={total_cgcg}):")
print(f"    • Identically methylated:     {identical_count:>5} ({pct_identical:.2f}%)")
print(f"    • Differentially methylated:  {diff_count:>5} ({pct_diff:.2f}%)")

print("\n" + "=" * 60)
print("Summary: BPE tokenization splits CG sites in ~{:.1f}% of reads.".format(pct_cg_split))
print("         Multi-CG tokens are rare ({:.2f}%) and mostly concordant.".format(pct_two_cgs))
print("=" * 60 + "\n")


Counting CG patterns: 100%|██████████| 258933/258933 [00:00<00:00, 299847.91it/s]



DNABERT2 BPE Tokenization Analysis: CG Site Preservation

[1] CG Dinucleotide Tokenization:
    • Reads with ≥1 token containing 'CG': 88.96%
    • Reads where CG is split across tokens: 11.04%

[2] Multi-CG Token Frequency:
    • Reads with ≥1 token containing 'CGCG' (2 consecutive CGs): 0.5863%
    • Reads with ≥2 tokens containing 'CGCG': 0.0000%
    • Reads with ≥1 token containing 'CGCGCG' (3 consecutive CGs): 0.0000%

[3] Methylation Status Concordance in CGCG Tokens (n=1518):
    • Identically methylated:      1425 (93.87%)
    • Differentially methylated:     93 (6.13%)

Summary: BPE tokenization splits CG sites in ~11.0% of reads.
         Multi-CG tokens are rare (0.59%) and mostly concordant.

